# Expense Tools Integration with ConversationAgent

This notebook demonstrates the integration of expense_tools with the ConversationAgent, showing:
- Sync tool usage with `dspy.Tool()`
- Async tool usage with `tool.acall()`
- Integration with Sofia conversation agent
- Concurrent async operations

## Setup and Configuration

In [1]:
import asyncio
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

import dspy
from dotenv import load_dotenv

from src.application.interfaces.conversation_agent import ConversationContext, Message
from src.config.settings import settings
from src.infrastructure.database.session import get_session_maker
from src.infrastructure.llm.conversation_agent import ConversationAgent
from src.infrastructure.llm.embedding.gemini_embedding_service import GeminiEmbeddingService
from src.infrastructure.llm.tools.expenses import (
    configure_expense_tools,
    expense_tools,
    set_user_context,
)

# Load environment
load_dotenv()

print("✅ Imports successful")

✅ Imports successful


/Users/xavierau/Code/python/whatsapp_chatbot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configure expense tools with dependencies
embedding_service = GeminiEmbeddingService(api_key=settings.gemini_api_key)
session_maker = get_session_maker()
configure_expense_tools(session_maker, embedding_service)
set_user_context("test-user-notebook")

print("✅ Expense tools configured")

✅ Expense tools configured


## Section 1: Direct Tool Usage (Sync)

Test calling expense_tools directly as a synchronous function.

In [3]:
# Create an expense using sync call
result = expense_tools(
    operation="create",
    amount=500.0,
    currency="HKD",
    description="Lunch at Italian restaurant",
    category="Food & Dining",
    tags=["lunch", "italian"],
    expense_date="2025-10-26",
)

print("\n📝 Create Expense Result:")
print(result)

2025-10-26 16:04:12,747 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2025-10-26 16:04:12,747 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-26 16:04:12,750 INFO sqlalchemy.engine.Engine select current_schema()
2025-10-26 16:04:12,750 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-26 16:04:12,754 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2025-10-26 16:04:12,754 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-10-26 16:04:12,756 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:12,759 INFO sqlalchemy.engine.Engine SELECT expenses.id AS expenses_id, expenses.user_id AS expenses_user_id, expenses.amount AS expenses_amount, expenses.currency AS expenses_currency, expenses.description AS expenses_description, expenses.category AS expenses_category, expenses.tags AS expenses_tags, expenses.expense_date AS expenses_expense_date, expenses.is_recurring AS expenses_is_recurring, expenses.recurrence_pattern AS expenses_recurrence_pattern, expe

## Section 2: DSPy Tool Wrapper

Test using `dspy.Tool()` wrapper for DSPy's standard approach.

In [4]:
# Wrap expense_tools as a DSPy tool
expense_tool = dspy.Tool(expense_tools)

print("✅ DSPy Tool created")
print(f"Tool name: {expense_tool.name}")
print(f"Tool description: {expense_tool.desc}")

✅ DSPy Tool created
Tool name: expense_tools
Tool description: Universal expense tracking tool for LLM agent.

This is the single entry point for all expense operations. It validates
inputs and routes to appropriate operation handlers.

Operations:
- create: Create new expense (amount, description, category, currency?, tags?, expense_date?, is_recurring?, recurrence_pattern?, receipt_path?)
- update: Update existing expense (expense_id, amount?, currency?, description?, category?, tags?, expense_date?, is_recurring?, recurrence_pattern?, receipt_path?)
- delete: Delete expense (expense_id)
- get: Get single expense (expense_id)
- list: List expenses (limit?, offset?, date_from?, date_to?)
- search: Search expenses (query?, search_type, category?, tags?, amount_min?, amount_max?, limit?)
- analytics: Get analytics (analytics_type, year?, month?, date_from?, date_to?, category?)

Args:
    operation: Operation to perform
    **kwargs: Operation-specific arguments

Returns:
    Operation 

In [5]:
# Use the DSPy tool wrapper for sync call
result = expense_tool(
    operation="list",
    limit=5,
)

print("\n📋 List Expenses Result:")
print(result)

2025-10-26 16:04:12,869 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:12,870 INFO sqlalchemy.engine.Engine SELECT expenses.id, expenses.user_id, expenses.amount, expenses.currency, expenses.description, expenses.category, expenses.tags, expenses.expense_date, expenses.is_recurring, expenses.recurrence_pattern, expenses.receipt_path, expenses.embedding, expenses.created_at, expenses.updated_at 
FROM expenses 
WHERE expenses.user_id = $1::VARCHAR ORDER BY expenses.expense_date DESC, expenses.created_at DESC 
 LIMIT $2::INTEGER OFFSET $3::INTEGER
2025-10-26 16:04:12,870 INFO sqlalchemy.engine.Engine [generated in 0.00037s] ('test-user-notebook', 5, 0)
2025-10-26 16:04:12,875 INFO sqlalchemy.engine.Engine ROLLBACK

📋 List Expenses Result:
Found 5 expense(s):

1. [deff4b74-5605-49b0-a092-1c9a70851b3b] Lunch at Italian restaurant
   Amount: 500.00 HKD
   Category: Food & Dining
   Date: 2025-10-26
   Tags: lunch, italian

2. [2f2b8c53-948b-4a04-b164-dc7169ccd2b9] groceries 

## Section 3: Async Tool Usage

Test async tool calls using `tool.acall()` for high-performance scenarios.

In [6]:
# Test async tool call
result = await expense_tool.acall(
    operation="create",
    amount=350.0,
    currency="HKD",
    description="Coffee and pastries",
    category="Food & Dining",
    tags=["coffee", "breakfast"],
    expense_date="2025-10-26",
)

print("☕ Async Create Result:")
print(result)

2025-10-26 16:04:13,440 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:13,441 INFO sqlalchemy.engine.Engine SELECT expenses.id AS expenses_id, expenses.user_id AS expenses_user_id, expenses.amount AS expenses_amount, expenses.currency AS expenses_currency, expenses.description AS expenses_description, expenses.category AS expenses_category, expenses.tags AS expenses_tags, expenses.expense_date AS expenses_expense_date, expenses.is_recurring AS expenses_is_recurring, expenses.recurrence_pattern AS expenses_recurrence_pattern, expenses.receipt_path AS expenses_receipt_path, expenses.embedding AS expenses_embedding, expenses.created_at AS expenses_created_at, expenses.updated_at AS expenses_updated_at 
FROM expenses 
WHERE expenses.id = $1::VARCHAR
2025-10-26 16:04:13,441 INFO sqlalchemy.engine.Engine [cached since 0.6818s ago] ('1093d3bf-51f3-45b8-92e7-ec8bdb55f1c2',)
2025-10-26 16:04:13,444 INFO sqlalchemy.engine.Engine INSERT INTO expenses (id, user_id, amount, currenc

## Section 4: Concurrent Async Operations

Demonstrate parallel async operations for high throughput.

In [7]:
import time

print("⚡ Testing concurrent async operations...\n")

start_time = time.time()

# Create multiple expenses concurrently
tasks = [
    expense_tool.acall(
        operation="create",
        amount=200.0,
        currency="HKD",
        description="Taxi ride to office",
        category="Transportation",
        tags=["taxi", "work"],
        expense_date="2025-10-26",
    ),
    expense_tool.acall(
        operation="create",
        amount=850.0,
        currency="HKD",
        description="Dinner with team",
        category="Food & Dining",
        tags=["dinner", "team"],
        expense_date="2025-10-26",
    ),
    expense_tool.acall(
        operation="create",
        amount="450.0",
        currency="HKD",
        description="Book purchase",
        category="Education",
        tags=["books", "learning"],
        expense_date="2025-10-26",
    ),
]

# Execute all tasks concurrently
results = await asyncio.gather(*tasks)

elapsed_time = time.time() - start_time

print(f"\n✅ Created {len(results)} expenses concurrently in {elapsed_time:.2f}s")
for i, result in enumerate(results, 1):
    print(f"\nExpense {i}:")
    print(result[:100] + "..." if len(result) > 100 else result)

⚡ Testing concurrent async operations...

2025-10-26 16:04:14,879 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:14,879 INFO sqlalchemy.engine.Engine SELECT expenses.id AS expenses_id, expenses.user_id AS expenses_user_id, expenses.amount AS expenses_amount, expenses.currency AS expenses_currency, expenses.description AS expenses_description, expenses.category AS expenses_category, expenses.tags AS expenses_tags, expenses.expense_date AS expenses_expense_date, expenses.is_recurring AS expenses_is_recurring, expenses.recurrence_pattern AS expenses_recurrence_pattern, expenses.receipt_path AS expenses_receipt_path, expenses.embedding AS expenses_embedding, expenses.created_at AS expenses_created_at, expenses.updated_at AS expenses_updated_at 
FROM expenses 
WHERE expenses.id = $1::VARCHAR
2025-10-26 16:04:14,880 INFO sqlalchemy.engine.Engine [cached since 2.121s ago] ('63a2d5e1-6f69-4547-939b-83bbcf5e76eb',)
2025-10-26 16:04:14,883 INFO sqlalchemy.engine.Engine INSERT IN

## Section 5: Integration with ConversationAgent

Test expense tools working seamlessly with Sofia conversation agent.

In [8]:
lm = dspy.LM("gemini/gemini-2.5-flash", api_key=settings.gemini_api_key, max_tokens=20000)
dspy.settings.configure(lm=lm)

# Initialize conversation agent
agent = ConversationAgent(model=settings.default_llm_model)

print("✅ ConversationAgent initialized with expense tools")
print(f"Model: {agent.model}")
print(f"Temperature: {agent.temperature}")

✅ ConversationAgent initialized with expense tools
Model: gemini/gemini-2.5-flash
Temperature: 0.7


In [9]:
# Test Sofia creating an expense through conversation
context = ConversationContext(
    current_message="I just spent 680 HKD on groceries at the supermarket. Can you track this expense for me?",
    metadata={"user_id": "test-user-notebook", "session_id": "notebook-session"},
)

response = agent.generate_response(context)

print("\n🛒 Sofia's Response to Grocery Expense:")
print(response.content)

Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_span() or that you're operating within an active span context.


2025-10-26 16:04:15,679 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:15,679 INFO sqlalchemy.engine.Engine SELECT expenses.id AS expenses_id, expenses.user_id AS expenses_user_id, expenses.amount AS expenses_amount, expenses.currency AS expenses_currency, expenses.description AS expenses_description, expenses.category AS expenses_category, expenses.tags AS expenses_tags, expenses.expense_date AS expenses_expense_date, expenses.is_recurring AS expenses_is_recurring, expenses.recurrence_pattern AS expenses_recurrence_pattern, expenses.receipt_path AS expenses_receipt_path, expenses.embedding AS expenses_embedding, expenses.created_at AS expenses_created_at, expenses.updated_at AS expenses_updated_at 
FROM expenses 
WHERE expenses.id = $1::VARCHAR
2025-10-26 16:04:15,680 INFO sqlalchemy.engine.Engine [cached since 2.921s ago] ('a377a71f-af4b-4d67-b8db-a8d463e72c26',)
2025-10-26 16:04:15,683 INFO sqlalchemy.engine.Engine INSERT INTO expenses (id, user_id, amount, currency

In [10]:
# Test Sofia searching expenses
context = ConversationContext(
    current_message="Can you show me all my food-related expenses?",
    history=(
        Message(role="user", content="I just spent 680 HKD on groceries at the supermarket."),
        Message(role="assistant", content=response.content),
    ),
    metadata={"user_id": "test-user-notebook", "session_id": "notebook-session"},
)

response = agent.generate_response(context)

print("\n🍽️ Sofia's Response to Food Expense Search:")
print(response.content)

Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_span() or that you're operating within an active span context.
Error executing search: Text cannot be empty
Traceback (most recent call last):
  File "/Users/xavierau/Code/python/whatsapp_chatbot/src/infrastructure/llm/tools/expenses.py", line 869, in expense_tools
    result = asyncio.run(_execute())
  File "/Users/xavierau/Code/python/whatsapp_chatbot/.venv/lib/python3.13/site-packages/nest_asyncio.py", line 30, in run
    return loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/Users/xavierau/Code/python/whatsapp_chatbot/.venv/lib/python3.13/site-packages/nest_asyncio.py", line 98, in run_until_complete
    return f.result()
           ~~~~~~~~^^
  File "/Users/xavierau/.local/share/uv/python/cpython-3.13.2-macos-aarch64-none/lib/python3.13/asyncio/futures.py", line 199, in result
    raise self._exce

2025-10-26 16:04:23,745 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:23,749 INFO sqlalchemy.engine.Engine SELECT expenses.id, expenses.user_id, expenses.amount, expenses.currency, expenses.description, expenses.category, expenses.tags, expenses.expense_date, expenses.is_recurring, expenses.recurrence_pattern, expenses.receipt_path, expenses.embedding, expenses.created_at, expenses.updated_at, $1::INTEGER - (expenses.embedding <=> $2) AS similarity 
FROM expenses 
WHERE expenses.user_id = $3::VARCHAR AND $1::INTEGER - (expenses.embedding <=> $2) >= $4::FLOAT ORDER BY similarity DESC 
 LIMIT $5::INTEGER
2025-10-26 16:04:23,750 INFO sqlalchemy.engine.Engine [generated in 0.00194s] (1, '[-0.003989289049059153,0.0006115392898209393,0.01674295961856842,-0.08151896297931671,0.008429848589003086,0.0007064097444526851,-0.01185029093176126 ... (32959 characters truncated) ... 006595865823328495,-0.005109020043164492,-0.005028742831200361,-0.007005095016211271,0.004146216902881

In [11]:
# Test Sofia providing expense analytics
context = ConversationContext(
    current_message="Can you give me a summary of my spending by category?",
    metadata={"user_id": "test-user-notebook", "session_id": "notebook-session"},
)

response = agent.generate_response(context)

print("\n📊 Sofia's Response to Analytics Request:")
print(response.content)

Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_span() or that you're operating within an active span context.
Validation failed: Error: 'analytics_type' is required. Valid types: total_by_category, monthly_summary.
Validation failed: Error: 'analytics_type' is required. Valid types: total_by_category, monthly_summary.


2025-10-26 16:04:28,875 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:28,876 INFO sqlalchemy.engine.Engine SELECT expenses.category, sum(expenses.amount) AS total 
FROM expenses 
WHERE expenses.user_id = $1::VARCHAR GROUP BY expenses.category ORDER BY total DESC
2025-10-26 16:04:28,876 INFO sqlalchemy.engine.Engine [generated in 0.00033s] ('test-user-notebook',)
2025-10-26 16:04:28,882 INFO sqlalchemy.engine.Engine ROLLBACK
response_content Here's a summary of your spending by category! 📊

*   **Food & Dining:** 5100.00 HKD
*   **Work Equipment:** 2500.00 HKD
*   **Groceries:** 2040.00 HKD
*   **Education:** 900.00 HKD
*   **Transportation:** 400.00 HKD

Let me know if you'd like to dive deeper into any of these categories or see a summary for a different period! 😊

📊 Sofia's Response to Analytics Request:
Here's a summary of your spending by category! 📊

*   **Food & Dining:** 5100.00 HKD
*   **Work Equipment:** 2500.00 HKD
*   **Groceries:** 2040.00 HKD
*   **Educat

## Section 6: Async ConversationAgent

Test async response generation for high-performance scenarios.

In [12]:
print("🚀 Testing async conversation agent...\n")

context = ConversationContext(
    current_message="I spent 2500 HKD on a new laptop for work. Please add this as a work equipment expense.",
    metadata={"user_id": "test-user-notebook", "session_id": "notebook-session"},
)

response = await agent.agenerate_response(context)

print("💻 Async Sofia's Response:")
print(response.content)
print(f"\nAsync: {response.metadata.get('async', False)}")

Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_span() or that you're operating within an active span context.


🚀 Testing async conversation agent...

2025-10-26 16:04:33,440 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-10-26 16:04:33,441 INFO sqlalchemy.engine.Engine SELECT expenses.id AS expenses_id, expenses.user_id AS expenses_user_id, expenses.amount AS expenses_amount, expenses.currency AS expenses_currency, expenses.description AS expenses_description, expenses.category AS expenses_category, expenses.tags AS expenses_tags, expenses.expense_date AS expenses_expense_date, expenses.is_recurring AS expenses_is_recurring, expenses.recurrence_pattern AS expenses_recurrence_pattern, expenses.receipt_path AS expenses_receipt_path, expenses.embedding AS expenses_embedding, expenses.created_at AS expenses_created_at, expenses.updated_at AS expenses_updated_at 
FROM expenses 
WHERE expenses.id = $1::VARCHAR
2025-10-26 16:04:33,441 INFO sqlalchemy.engine.Engine [cached since 20.68s ago] ('3faf2bbd-7d38-4cab-b26a-93a0f08c4848',)
2025-10-26 16:04:33,446 INFO sqlalchemy.engine.Engine INSERT INTO 

## Summary

This notebook demonstrated:

1. ✅ **Direct sync usage**: `expense_tools(operation, **kwargs)`
2. ✅ **DSPy Tool wrapper**: `dspy.Tool(expense_tools)`
3. ✅ **Async tool calls**: `await expense_tool.acall(**kwargs)`
4. ✅ **Concurrent operations**: `asyncio.gather(*tasks)`
5. ✅ **ConversationAgent sync**: `agent.generate_response(context)`
6. ✅ **ConversationAgent async**: `await agent.agenerate_response(context)`

The expense_tools are now fully integrated with the conversation agent and support both sync and async operation modes!